In [ ]:
import os
import shutil
import sys

In [ ]:
def delete_checkpoints(root: str,
                       prefix: str = "checkpoint",
                       case_sensitive: bool = False,
                       dry_run: bool = True,
                       follow_symlinks: bool = False) -> int:
    """
    Recursively delete directories whose name starts with `prefix`.

    Returns the count of directories matched (deleted or would delete if dry-run).
    """
    if not os.path.isdir(root):
        raise NotADirectoryError(f"Root path is not a directory: {root}")

    matched = 0
    # Walk bottom-up so we can remove directories safely
    for cur_root, dirs, _files in os.walk(root, topdown=False, followlinks=follow_symlinks):
        for d in list(dirs):
            name_to_check = d if case_sensitive else d.lower()
            prefix_to_check = prefix if case_sensitive else prefix.lower()

            if name_to_check.startswith(prefix_to_check):
                full_path = os.path.join(cur_root, d)

                # Skip if it's a symlink (unless explicitly following)
                if not follow_symlinks and os.path.islink(full_path):
                    print(f"SKIP (symlink): {full_path}")
                    continue

                matched += 1
                if dry_run:
                    print(f"DRY-RUN would delete: {full_path}")
                else:
                    try:
                        shutil.rmtree(full_path)
                        print(f"Deleted: {full_path}")
                    except Exception as e:
                        print(f"ERROR deleting {full_path}: {e}", file=sys.stderr)
    return matched

In [ ]:
root = "./results/finetuning/"

In [ ]:
delete_checkpoints(root=root, dry_run=True)

In [ ]:
delete_checkpoints(root=root, dry_run=False)